In [1]:
EXPERIMENTAL_ID="search_weighted_feature_rerank"
START_DATE="2026-03-05"

In [2]:
from odps_client import get_odps_sql_result_as_df
from datetime import datetime, timedelta

daily_high_search_volume_theshold = 470 / 14
daily_low_search_volume_theshold = 2

last_n_days = 60
ds_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
last_n_days_ago = (datetime.now() - timedelta(days=last_n_days)).strftime("%Y%m%d")

        #         ,CASE
        #     WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > {last_n_days*daily_high_search_volume_theshold} THEN '高频搜索词'
        #     WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) <= {last_n_days*daily_low_search_volume_theshold} THEN '低频搜索词'
        #     ELSE '中频搜索词'
        #  END

top_query = f"""
SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS std_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.5) AS p50_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.75) AS p75_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.9) AS p90_click_index
        ,RANK() OVER (PARTITION BY "dontcarte" ORDER BY COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) DESC) as search_cnt_rnk
        ,"不区分频次" AS 搜索频次标签
        ,COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) as ctr_uv
        ,COUNT(distinct ds) as 有搜索天数
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > 0.25 THEN '高点击率词'
            ELSE '低点击率词'
         END AS 点击率标签
FROM    summerfarm_tech.app_log_search_detail_di
WHERE   ds BETWEEN '{last_n_days_ago}' and '{ds_yesterday}'
GROUP BY query
ORDER BY searched_users DESC;
"""

top_query_df = get_odps_sql_result_as_df(sql=top_query)
top_query_df['搜索频次标签']=top_query_df['search_cnt_rnk'].apply(lambda x: 'top400' if x <= 400 else 'top400以外')
top_query_df.head(20)

2026-03-22 21:12:40 - INFO - Thread count: 20
2026-03-22 21:12:58 - INFO - Tunnel session created: <InstanceDownloadSession id=2026032221125733d5c20b2beb2b16 project_name=summerfarm_ds instance_id=20260322131240517ggp15vu7q6t6>
2026-03-22 21:13:00 - INFO - sql:

SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS 

,query,searched_users,search_cnt,click_cnt,avg_click_index,max_click_index,min_click_index,std_click_index,p50_click_index,p75_click_index,p90_click_index,search_cnt_rnk,搜索频次标签,ctr_uv,有搜索天数,点击率标签
0,芒果,11792,231101,61644,6.6,143.0,0.0,7.1,5.0,10.0,14.0,1,top400,0.266741,60,高点击率词
1,牛奶,11693,131477,45736,4.4,156.0,0.0,7.0,1.0,6.0,12.0,3,top400,0.347863,60,高点击率词
2,草莓,11224,196262,64264,4.8,158.0,0.0,8.3,2.0,5.0,15.0,2,top400,0.327440,60,高点击率词
3,安佳,8750,75713,23768,1.7,205.0,0.0,4.9,0.0,1.0,6.0,7,top400,0.313922,60,高点击率词
4,蓝莓,8463,101228,29383,2.2,197.0,0.0,3.8,1.0,3.0,5.0,5,top400,0.290266,60,高点击率词
5,奶油,8367,104670,24849,11.0,162.0,0.0,17.3,3.0,16.0,36.0,4,top400,0.237403,60,低点击率词
6,柠檬,7079,76901,23073,4.5,672.0,0.0,12.6,2.0,6.0,12.0,6,top400,0.300035,60,高点击率词
7,黄油,6913,60499,11652,8.1,165.0,0.0,11.8,3.0,12.0,21.0,8,top400,0.192598,60,低点击率词
8,铁塔,6374,51722,14616,0.5,286.0,0.0,5.9,0.0,0.0,0.0,9,top400,0.282588,60,高点击率词
9,安佳淡奶油,5444,36798,12851,1.0,175.0,0.0,5.4,0.0,0.0,2.0,15,top400,0.349231,60,高点击率词


In [3]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd

# 设置pandas显示选项以展示更多内容
pd.set_option("display.max_rows", 100)  # 显示最多100行
pd.set_option("display.max_columns", None)  # 显示所有列
pd.set_option("display.width", 1000)  # 设置显示宽度
pd.set_option("display.max_colwidth", 100)  # 设置列最大宽度

import sqlite3


def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    """
    从SLS(Simple Log Service)获取指定日期的用户变体数据。

    Args:
        day (datetime): 要获取数据的日期。
        check_if_local_exist (bool): 是否检查本地数据库中是否已存在数据，默认为True。

    Returns:
        pd.DataFrame: 包含用户变体数据的DataFrame。
    """
    # 构建数据库文件名和表名
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    # 连接到SQLite数据库
    conn = sqlite3.connect(db_file_name)

    # 如果设置为检查本地数据
    if check_if_local_exist:
        try:
            # 尝试从数据库中读取数据
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            # 关闭数据库连接
            conn.close()
            # 返回读取的数据
            return df
        except pd.io.sql.DatabaseError:
            # 如果表不存在，则忽略错误
            pass

    # 构建SLS查询语句
    query = f"""
type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{{digit}}') as api,
    pageName as page_name,
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id,
    type,
    uid,
    date_format(__time__, '%Y%m%d') as ds,
    count(1) as search_times,
    array_join(array_sort(array_agg(distinct regexp_extract(experiment_item, '"variantId":"([^"]+)"', 1))),',') as variant_list
FROM log, 
UNNEST(regexp_extract_all(json_extract_scalar(ai, '$.qh.xm-ab-exp'), '\{{[^}}]+\}}')) as t(experiment_item)
WHERE experiment_item LIKE '%"experimentId":"{EXPERIMENTAL_ID}"%'
GROUP BY 1,2,3,4,5,6
LIMIT 1000000
"""
    # 设置查询的起始时间和结束时间
    print(query)
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    # 从SLS获取数据
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",  # 指定SLS项目
        logstore="xm-mall",  # 指定SLS日志库
        from_time=from_time,  # 指定查询起始时间
        to_time=to_time,  # 指定查询结束时间
    )

    # 将search_times列中的缺失值填充为1，并转换为整数类型
    _df["search_times"] = _df["search_times"].fillna(1).astype(int)
    # 将variant_list列中的缺失值填充为"none"
    _df["variant_list"] = _df["variant_list"].fillna("none")

    # 如果DataFrame不为空
    if not _df.empty:
        # 删除不需要的列
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        # 将数据写入SQLite数据库，如果表已存在则替换
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    # 关闭数据库连接
    conn.close()
    # 返回数据
    return _df


# 创建一个空的DataFrame来存储所有日期的用户变体数据
all_user_variant_df = pd.DataFrame()
# 设置起始日期和结束日期
start_date = datetime.strptime(START_DATE, "%Y-%m-%d")
end_date = datetime.now()
# 从起始日期开始循环，直到结束日期
current_date = start_date
while current_date <= end_date:
    # 检查是否是今天
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    # 如果是今天,则跳过，因为今天的数据可能不完整
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    # 获取当前日期的用户变体数据
    df = get_user_variant_of_date_from_sls(current_date, check_if_local_exist=True)
    # 将当前日期的数据添加到总的DataFrame中
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    # 日期增加一天
    current_date += timedelta(days=1)

# 显示前10行数据
all_user_variant_df.head(10)


type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{digit}') as api,
    pageName as page_name,
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id,
    type,
    uid,
    date_format(__time__, '%Y%m%d') as ds,
    count(1) as search_times,
    array_join(array_sort(array_agg(distinct regexp_extract(experiment_item, '"variantId":"([^"]+)"', 1))),',') as variant_list
FROM log, 
UNNEST(regexp_extract_all(json_extract_scalar(ai, '$.qh.xm-ab-exp'), '\{[^}]+\}')) as t(experiment_item)
WHERE experiment_item LIKE '%"experimentId":"search_weighted_feature_rerank"%'
GROUP BY 1,2,3,4,5,6
LIMIT 1000000

即将获取数据: =====> 2026-03-15 00:00:00 2026-03-15 23:59:59.999999 xm-mall: 
type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+',
>=====数条数:7231

type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{digit}') as api,
    pageName

,api,page_name,experiment_id,type,uid,ds,search_times,variant_list
0,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,575888,20260305,1,V4
1,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,231161,20260305,2,V1
2,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,416895,20260305,2,V4
3,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,612996,20260305,4,V1
4,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,101709,20260305,1,V4
5,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,606468,20260305,1,V4
6,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,576717,20260305,1,V2
7,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,493584,20260305,5,V2
8,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,523138,20260305,3,V3
9,/mall/sku/page,/search/goods-new,search_weighted_feature_rerank,a,515082,20260305,1,V4


In [4]:
import pandasql
stats=pandasql.sqldf("""select ds,variant_list,count(distinct uid) unique_user 
                     from all_user_variant_df group by ds,variant_list order by ds desc,variant_list""")

display(stats)

,ds,variant_list,unique_user
0,20260321,V1,1872
1,20260321,V2,1920
2,20260321,V3,1865
3,20260321,V4,1892
4,20260320,V1,2132
5,20260320,V2,2102
6,20260320,V3,2110
7,20260320,V4,2063
8,20260319,V1,2052
9,20260319,V2,2056


In [5]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query = """
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,search_query,type,uid,ds
from(
select uid,date_format(__time__, '%Y%m%d') ds,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""


def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_view_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_view_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

即将获取数据: =====> 2026-03-15 00:00:00 2026-03-15 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:158376
即将获取数据: =====> 2026-03-16 00:00:00 2026-03-16 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:175307
即将获取数据: =====> 2026-03-17 00:00:00 2026-03-17 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:175045
即将获取数据: =====> 2026-03-18 00:00:00 2026-03-18 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:175177
即将获取数据: =====> 2026-03-19 00:00:00 2026-03-19 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:189319
即将获取数据: =====> 2026-03-20 00:00:00 2026-03-20 23:59:59.

In [6]:
from sls_client import get_sls_raw_data_by_query

click_query = """
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_item, 'name:([^,]+)', 1)) AS name,
  coalesce(sku,regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1)) AS sku,
  coalesce(pid,regexp_extract(sku_item, 'pid:([^,]+)', 1)) AS pid,
  coalesce(pdid,regexp_extract(sku_item, 'pdid:(\d+)', 1)) AS pdid,bid,
ds,search_query,type,uid,page_name,sku_item,coalesce(linkInfo,url)linkInfo from(
select uid,date_format(__time__, '%Y%m%d') ds,replace(replace(split_part(url_decode(split_part(url,'#/',2)),'?',2),'=',':'),'&',',') url,
bid_list.sku_item,pageName as page_name,bid,idx,name,sku,pid,pdid,linkInfo,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 1000000)"""

# click_query = "type:cl and pageName:/search/goods"


def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_raw_data_by_query(
        query=click_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(
            columns=[
                "__source__",
                "__time__",
                "userAgent",
                "url",
                "__topic__",
                "__tag__:__client_ip__",
                "__tag__:__receive_time__",
                "__time_ns_part__",
            ],
            inplace=True,
            errors="ignore",
        )
        _df["ds"] = day.strftime("%Y%m%d")
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_click_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_click_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

即将获取数据: =====>from_time:2026-03-15 00:00:00, to_time:2026-03-15 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:35069
即将获取数据: =====>from_time:2026-03-16 00:00:00, to_time:2026-03-16 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:35953
即将获取数据: =====>from_time:2026-03-17 00:00:00, to_time:2026-03-17 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:36549
即将获取数据: =====>from_time:2026-03-18 00:00:00, to_time:2026-03-18 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extra

,idx,name,sku,pid,pdid,bid,ds,search_query,type,uid,page_name,sku_item,linkInfo
0,3,小蜜哈密瓜 净重5-6斤/一级/2个,5485336704,唤起购买,1316,"{""categoryIdList"":[]}",20260305,哈密瓜,cl,291082,/search/goods-new,"{""categoryIdList"":[]}","name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no"
1,0,三象水磨糯米粉 500g*20包,3807662084,唤起购买,960,"{""categoryIdList"":[]}",20260305,三象水磨糯米粉,cl,610325,/search/goods-new,"{""categoryIdList"":[]}","name:Search,word:智利牛油果特惠,linkShadingWord:[object Object],isTiming:no"
2,1,爱乐薇(铁塔)淡奶油 1L*12盒,N001S01R002,唤起购买,52,"{""categoryIdList"":[]}",20260305,奶油,cl,97249,/search/goods-new,"{""categoryIdList"":[]}","name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no"
3,null,爱乐薇(铁塔)淡奶油 1L*12盒,N001S01R002,加购弹窗,52,"name:爱乐薇(铁塔)淡奶油 1L*12盒,pid:加购弹窗,sku:N001S01R002,pdid:52,stock:209",20260305,奶油,cl,97249,/search/goods-new,"name:爱乐薇(铁塔)淡奶油 1L*12盒,pid:加购弹窗,sku:N001S01R002,pdid:52,stock:209","name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no"
4,null,加入购物车,3807662084,加购弹窗,960,"name:加入购物车,pid:加购弹窗,sku:3807662084,pdid:960,stock:5",20260305,三象水磨糯米粉,cl,610325,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:3807662084,pdid:960,stock:5","name:Search,word:智利牛油果特惠,linkShadingWord:[object Object],isTiming:no"
5,null,步进器,1256524478,加购弹窗,null,undefined,20260305,芝士片,cl,364754,/search/goods-new,undefined,"name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no"
6,null,加入购物车,1256524478,加购弹窗,719,"name:加入购物车,pid:加购弹窗,sku:1256524478,pdid:719,stock:34",20260305,芝士片,cl,364754,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:1256524478,pdid:719,stock:34","name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no"
7,null,加入常购,608711118847,商品列表,null,"name:加入常购,sku:608711118847,skuName:悦鲜活牛奶 950mL*12瓶,source:2,pid:商品列表",20260305,悦鲜活,cl,500160,/search/goods-new,"name:加入常购,sku:608711118847,skuName:悦鲜活牛奶 950mL*12瓶,source:2,pid:商品列表","name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no"
8,null,移除常购,608711118847,商品列表,null,"name:移除常购,sku:608711118847,skuName:悦鲜活牛奶 950mL*12瓶,source:2,pid:商品列表",20260305,悦鲜活,cl,500160,/search/goods-new,"name:移除常购,sku:608711118847,skuName:悦鲜活牛奶 950mL*12瓶,source:2,pid:商品列表","name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no"
9,0,悦鲜活牛奶 950mL*12瓶,608711118847,goods,3028,"idx:0,name:悦鲜活牛奶 950mL*12瓶,pid:goods,sku:608711118847,salePrice:145,pdid:3028,stock:10000,ext:cross",20260305,悦鲜活,cl,500160,/search/goods-new,"idx:0,name:悦鲜活牛奶 950mL*12瓶,pid:goods,sku:608711118847,salePrice:145,pdid:3028,stock:10000,ext:cross","name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no"


In [7]:
import re

all_user_sku_click_explored = []
pattern = re.compile(r'idx:(?P<idx>\d+).*?name:(?P<name>[^,]+).*?pid:(?P<pid>[^,]+).*?sku:(?P<sku>[^,]+).*?pdid:(?P<pdid>[^,]+)')

for index, row in all_user_sku_click_df.iterrows():
    _dict = row.to_dict()
    pid = _dict['pid']
    sku = _dict['sku']
    search_query=_dict["search_query"]
    if not search_query or f"{search_query}" == "":
        search_query = _dict["linkInfo"]
        # 搜索pdName，如果没找到，则search_query为空字符串
        match = re.search(r'pdName:([^,]+)', search_query)
        search_query = match.group(1) if match else ""
    bid = _dict["bid"]
    for bid_item in bid.split(";"):
        sku_info = {}
        sku_info.update(_dict)
        sku_info["search_query"] = search_query
        sku_info["bid"] = bid_item
        if 'undefined' in bid_item:
            all_user_sku_click_explored.append(sku_info)
        else:
            try:
                idx = pdid = name = None
                
                idx_match = re.search(r'idx:(\d+)', bid_item)
                if idx_match:
                    idx = idx_match.group(1)
                    
                pdid_match = re.search(r'pdid:(\d+)', bid_item)
                if pdid_match:
                    pdid = pdid_match.group(1)
                    
                sku_match = re.search(r'sku:([\dA-Za-z]+)', bid_item)
                if sku_match:
                    sku = sku_match.group(1)
                    
                pid_match = re.search(r'pid:([^,]+)', bid_item)
                if pid_match:
                    pid = pid_match.group(1)
                    
                name_match = re.search(r'name:([^,]+)', bid_item)
                if name_match:
                    name = name_match.group(1)
                    
                sku_info.update({
                    "idx": idx,
                    "pdid": pdid, 
                    "sku": sku,
                    "pid": pid,
                    "name": name
                })
                all_user_sku_click_explored.append(sku_info)
            except Exception as e:
                print(e, bid_item)
                raise e

all_user_sku_click_explored_df = pd.DataFrame(all_user_sku_click_explored)
all_user_sku_click_explored_df[['bid','sku','name','idx','pid','pdid','linkInfo','search_query']].head(5)

,bid,sku,name,idx,pid,pdid,linkInfo,search_query
0,"{""categoryIdList"":[]}",5485336704,None,None,唤起购买,None,"name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no",哈密瓜
1,"{""categoryIdList"":[]}",3807662084,None,None,唤起购买,None,"name:Search,word:智利牛油果特惠,linkShadingWord:[object Object],isTiming:no",三象水磨糯米粉
2,"{""categoryIdList"":[]}",N001S01R002,None,None,唤起购买,None,"name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no",奶油
3,"name:爱乐薇(铁塔)淡奶油 1L*12盒,pid:加购弹窗,sku:N001S01R002,pdid:52,stock:209",N001S01R002,爱乐薇(铁塔)淡奶油 1L*12盒,None,加购弹窗,52,"name:Search,word:PT生椰乳新品上市,linkShadingWord:[object Object],isTiming:no",奶油
4,"name:加入购物车,pid:加购弹窗,sku:3807662084,pdid:960,stock:5",3807662084,加入购物车,None,加购弹窗,960,"name:Search,word:智利牛油果特惠,linkShadingWord:[object Object],isTiming:no",三象水磨糯米粉


In [8]:
print(all_user_variant_df.columns)
print(all_user_sku_view_df.columns)
print(all_user_sku_click_df.columns)

all_sku_view_data_df = all_user_sku_view_df[
    [
        "idx",
        "name",
        "sku",
        "pid",
        "pdid",
        "uid",
        "ds",
        "search_query",
        "type",
    ]
].merge(
    all_user_variant_df[["uid", "ds", "variant_list", "search_times"]],
    on=["uid", "ds"],
    how="left",
)


Index(['api', 'page_name', 'experiment_id', 'type', 'uid', 'ds', 'search_times', 'variant_list'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'search_query', 'type', 'uid', 'ds'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'bid', 'ds', 'search_query', 'type', 'uid', 'page_name', 'sku_item', 'linkInfo'], dtype='object')


In [9]:
all_user_sku_click_explored_df.groupby("pid").size().reset_index(name="count").sort_values(
    "count", ascending=False
).head(10)

,pid,count
7,加购弹窗,219784
8,唤起购买,182140
4,goods,126636
10,商品卡片,125260
9,商品列表,5817
16,横版筛选栏,5615
5,mini榜单,3207
1,AI采购,345
18,竖版筛选栏,322
2,AI问题,76


In [10]:
user_click_with_variant_df = all_user_sku_click_explored_df.merge(
    all_user_variant_df[["uid", "ds", "variant_list"]],
    on=["uid", "ds"],
    how="left",
)

user_click_with_variant_df["action_type"] = user_click_with_variant_df.apply(
    lambda row: (
        "加入购物车"
        if row["pid"] == "加购弹窗" and row["name"] == "加入购物车"
        else "商品详情" if row["pid"] == "goods" else row["pid"]
    ),
    axis=1,
)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].fillna(-1)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].replace('null', -1).astype(int)

In [11]:
import pandasql

user_click_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,variant_list,ds,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,
count(case when action_type='商品详情' then 1 end) as 商品详情cnt,
count(case when action_type='加入购物车' then 1 end) as 加入购物车cnt,
count(case when action_type='唤起购买' and variant_list is not null then 1 end) as 唤起购买cnt,
count(case when (action_type='唤起购买' and variant_list is not null) or action_type='商品详情' then 1 end) as 总点击cnt,
count(case when (action_type='唤起购买' and variant_list is not null or action_type='商品详情') and idx>=0 and idx<=5 then 1 end) as 首屏总点击cnt,
round(avg(case when action_type='商品详情' or action_type='唤起购买' then idx end),1) as avg点击位置,
coalesce(max(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as max点击位置,
coalesce(min(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as min点击位置,
count(distinct sku) 点击SKU_cnt,
count(distinct search_query) 搜索词cnt                        
from user_click_with_variant_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
""")

user_click_with_variant_statistics_df.head(5)

,uid,variant_list,ds,搜索频次标签,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt
0,10,V2,20260313,top400,0,1,1,1,0,-1.0,-1,-1,1,1
1,10,V2,20260319,top400,0,0,1,1,0,-1.0,-1,-1,1,1
2,100027,None,20260306,top400以外,0,1,0,0,0,-1.0,-1,-1,1,1
3,100027,V3,20260311,top400以外,0,1,1,1,0,-1.0,-1,-1,1,1
4,100027,V3,20260312,top400以外,0,2,2,2,0,-1.0,-1,-1,2,2


In [12]:
# all_sku_view_data_df
# user_click_with_variant_df
# top_query_df

query_level_analytics=f"""
select 
    a.variant_list,a.search_query,c.搜索频次标签,
    count(distinct a.uid) as uv,count(distinct b.uid) as 点击uv,
    count(1) as 查看cnt,count(case when b.uid is not null then 1 end) as 点击cnt
    ,round(1.00 * count(case when b.uid is not null then 1 end)/count(1),4) as 点击率
    ,max(c.searched_users) total_searched_users
from all_sku_view_data_df a
left join user_click_with_variant_df b on a.uid = b.uid and a.ds = b.ds and a.search_query = b.search_query and a.idx = b.idx
left join top_query_df c on a.search_query = c.query
group by a.variant_list,a.search_query,c.搜索频次标签
order by total_searched_users desc,a.variant_list
limit 1600
"""

query_level_analytics_df = pandasql.sqldf(query_level_analytics)
query_level_analytics_df.dropna(subset=["variant_list"], inplace=True)
query_level_analytics_df.head(20)

,variant_list,search_query,搜索频次标签,uv,点击uv,查看cnt,点击cnt,点击率,total_searched_users
1,V1,芒果,top400,1378,443,33091,1126,0.0340,11792
2,V2,芒果,top400,1415,455,34552,1298,0.0376,11792
3,V3,芒果,top400,1400,453,28776,1248,0.0434,11792
4,V4,芒果,top400,1380,475,26823,1171,0.0437,11792
6,V1,牛奶,top400,1266,440,24141,1251,0.0518,11693
7,V2,牛奶,top400,1301,456,26651,1467,0.0550,11693
8,V3,牛奶,top400,1187,399,21463,1062,0.0495,11693
9,V4,牛奶,top400,1220,435,23655,1249,0.0528,11693
11,V1,草莓,top400,1263,433,28463,1148,0.0403,11224
12,V2,草莓,top400,1267,440,27004,1188,0.0440,11224


In [13]:
query_level_analytics_df_v2=query_level_analytics_df[query_level_analytics_df["variant_list"] == "V2"]
query_level_analytics_df_v2=query_level_analytics_df_v2[["search_query","点击率"]]
query_level_analytics_df_v2.columns=["search_query","点击率_v2"]

query_level_analytics_df=query_level_analytics_df.merge(query_level_analytics_df_v2, on="search_query", how="left")
query_level_analytics_df["点击率_变化%"]=query_level_analytics_df.apply(lambda row: 100.00*(row["点击率"]/row["点击率_v2"] - 1) if row["点击率_v2"] else None, axis=1)
query_level_analytics_df["点击率_变化%"]=query_level_analytics_df["点击率_变化%"].round(2)
query_level_analytics_df.to_csv(f"./data/搜索AB-查询词粒度分析-top400-{START_DATE}.csv", index=False)
query_level_analytics_df.head(20)

,variant_list,search_query,搜索频次标签,uv,点击uv,查看cnt,点击cnt,点击率,total_searched_users,点击率_v2,点击率_变化%
0,V1,芒果,top400,1378,443,33091,1126,0.0340,11792,0.0376,-9.57
1,V2,芒果,top400,1415,455,34552,1298,0.0376,11792,0.0376,0.00
2,V3,芒果,top400,1400,453,28776,1248,0.0434,11792,0.0376,15.43
3,V4,芒果,top400,1380,475,26823,1171,0.0437,11792,0.0376,16.22
4,V1,牛奶,top400,1266,440,24141,1251,0.0518,11693,0.0550,-5.82
5,V2,牛奶,top400,1301,456,26651,1467,0.0550,11693,0.0550,0.00
6,V3,牛奶,top400,1187,399,21463,1062,0.0495,11693,0.0550,-10.00
7,V4,牛奶,top400,1220,435,23655,1249,0.0528,11693,0.0550,-4.00
8,V1,草莓,top400,1263,433,28463,1148,0.0403,11224,0.0440,-8.41
9,V2,草莓,top400,1267,440,27004,1188,0.0440,11224,0.0440,0.00


In [14]:
null_search_query_df = pandasql.sqldf(
"""select case when search_query is null or search_query = 'null' then 'null-search-query' else 'normal' end has_search_query,
                                count(1) cnt from all_sku_view_data_df group by 1"""
)
null_search_query_df
# 约有2.3%的数据没有搜索词，这部分需要过滤掉
all_sku_view_data_df = all_sku_view_data_df[
    all_sku_view_data_df["search_query"] != "null"
]

In [15]:
all_sku_view_data_df["idx"] = all_sku_view_data_df["idx"].fillna(-1).astype(int)

# 用来单独过滤某些词的表现
query_list=['牛奶','柠檬']
query_list_str='","'.join(query_list)


user_view_with_variant_statistics_df = pandasql.sqldf(
f"""
select uid,ds,variant_list,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,min(a.search_query) sample_query,
count(1) as 商品查看cnt,
count(distinct sku) as 查看SKU_cnt,
count(distinct search_query) as 查看搜索词cnt,
max(idx) as max查看位置,
max(search_times) as 搜索翻页数cnt
from all_sku_view_data_df a
left join top_query_df b on a.search_query = b.query
-- where a.search_query in ("{query_list_str}")
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
"""
)

print("unique sample_query:", user_view_with_variant_statistics_df['sample_query'].unique(), len(user_view_with_variant_statistics_df['sample_query'].unique()))

unique sample_query: ['低筋粉' '绿豆馅' '奶油奶酪' ... '植' '中性透明果膏' '巧克力脆脆'] 11002


In [16]:
first_view_sql=f"""
select variant_list,uid,search_query,count(*) as sku_viewed_cnt
from all_sku_view_data_df
where cast(idx as bigint)<=5
group by variant_list,uid,search_query
order by sku_viewed_cnt desc
"""

first_view_sql_df = pandasql.sqldf(first_view_sql)
first_view_sql_df



,variant_list,uid,search_query,sku_viewed_cnt
0,V2,17358,牛油果,199
1,V3,598780,树莓,176
2,V3,442486,芒果,170
3,V2,603714,牛奶,157
4,V4,51637,牛奶,152
...,...,...,...,...
231937,V4,83253,罗勒叶,1
231938,V4,85303,草莓,1
231939,V4,88398,王后日式,1
231940,V4,96011,咖奶浓缩奶,1


In [17]:
first_view_click_sql=f"""
select uid,search_query,count(*) as sku_clicked_cnt,min(idx),max(idx),avg(idx)
from all_user_sku_click_df
where idx is not null and idx != 'null' and cast(idx as bigint)<=5
group by uid,search_query
order by sku_clicked_cnt desc
"""

first_view_click_sql_df = pandasql.sqldf(first_view_click_sql)
first_view_click_sql_df



,uid,search_query,sku_clicked_cnt,min(idx),max(idx),avg(idx)
0,115275,耙耙柑,156,0,0,0.000000
1,421361,奶酪,144,0,5,0.090278
2,19037,君酪,142,0,1,0.014085
3,556625,羽衣甘蓝,131,0,0,0.000000
4,357349,羽衣甘蓝,106,0,0,0.000000
...,...,...,...,...,...,...
130398,99947,立高,1,0,0,0.000000
130399,99998,草莓,1,3,3,3.000000
130400,99999,扬雅,1,5,5,5.000000
130401,99999,牛奶,1,0,0,0.000000


In [18]:
first_view_merged_df=first_view_sql_df.merge(first_view_click_sql_df, on=["uid", "search_query"], how="left")
first_view_merged_df.head(5)

grouped_first_view_df=first_view_merged_df.groupby("search_query").agg({"sku_viewed_cnt": "sum", "sku_clicked_cnt": "sum"}).reset_index()
grouped_first_view_df.columns=["search_query","总浏览量","总点击量"]
grouped_first_view_df.head(5)

first_view_merged_df=first_view_merged_df.merge(grouped_first_view_df, on="search_query", how="left")
first_view_merged_df=first_view_merged_df.sort_values("总点击量", ascending=False)
first_view_merged_df["是否实验组"]=first_view_merged_df.apply(lambda x: "对照组" if x["variant_list"] in ["V1","V2"] else "实验组", axis=1)
first_view_merged_df.head(5)

first_view_merged_stats=pandasql.sqldf("""
select search_query,是否实验组, sum(sku_viewed_cnt) as 浏览量, sum(sku_clicked_cnt) as 点击量
    ,sum(sku_clicked_cnt)*1.00/sum(sku_viewed_cnt) as 点击率
    ,min(总浏览量) as 词的总浏览量
from first_view_merged_df 
group by 是否实验组,search_query
order by min(总浏览量) desc
""")

first_view_merged_stats.to_csv(f"./data/搜索AB-首屏点击分析-{START_DATE}.csv", index=False)
first_view_merged_stats.head(40)

,search_query,是否实验组,浏览量,点击量,点击率,词的总浏览量
0,芒果,实验组,31482,8785.0,0.279048,58952
1,芒果,对照组,27470,3455.0,0.125774,58952
2,草莓,实验组,30781,10872.0,0.353205,58522
3,草莓,对照组,27741,6767.0,0.243935,58522
4,牛奶,实验组,21702,6742.0,0.310663,42452
5,牛奶,对照组,20750,5036.0,0.242699,42452
6,蓝莓,实验组,20591,5865.0,0.284833,40177
7,蓝莓,对照组,19586,4649.0,0.237363,40177
8,安佳,实验组,16708,4575.0,0.273821,31408
9,安佳,对照组,14700,3140.0,0.213605,31408


In [19]:
controled_group_df=first_view_merged_stats[first_view_merged_stats["是否实验组"]=="对照组"]
controled_group_dict={}
for row_index, row in controled_group_df.iterrows():
    controled_group_dict[row["search_query"]]=row["点击率"]

first_view_merged_stats["点击率提升"]=first_view_merged_stats.apply(lambda row: 100.00*(row["点击率"]/controled_group_dict.get(row["search_query"],0.01) - 1), axis=1)
first_view_merged_stats["点击率提升"]=first_view_merged_stats["点击率提升"].round(2)
first_view_merged_stats.to_csv(f"./data/搜索AB-首屏点击分析-点击率提升-{START_DATE}.csv",index=False)
first_view_merged_stats.head(50)

,search_query,是否实验组,浏览量,点击量,点击率,词的总浏览量,点击率提升
0,芒果,实验组,31482,8785.0,0.279048,58952,121.87
1,芒果,对照组,27470,3455.0,0.125774,58952,0.00
2,草莓,实验组,30781,10872.0,0.353205,58522,44.79
3,草莓,对照组,27741,6767.0,0.243935,58522,0.00
4,牛奶,实验组,21702,6742.0,0.310663,42452,28.00
5,牛奶,对照组,20750,5036.0,0.242699,42452,0.00
6,蓝莓,实验组,20591,5865.0,0.284833,40177,20.00
7,蓝莓,对照组,19586,4649.0,0.237363,40177,0.00
8,安佳,实验组,16708,4575.0,0.273821,31408,28.19
9,安佳,对照组,14700,3140.0,0.213605,31408,0.00


In [20]:
first_view_merged_total_stats=pandasql.sqldf("""
select 是否实验组, sum(sku_viewed_cnt) as 浏览量, sum(sku_clicked_cnt) as 点击量
    ,sum(sku_clicked_cnt)*1.00/sum(sku_viewed_cnt) as 点击率
from first_view_merged_df 
group by 是否实验组
""")
first_view_merged_total_stats

,是否实验组,浏览量,点击量,点击率
0,实验组,827285,215640.0,0.260660
1,对照组,733415,162200.0,0.221157


In [21]:
all_data_df = user_view_with_variant_statistics_df.merge(
    user_click_with_variant_statistics_df, on=["uid", "ds", "variant_list","搜索频次标签"], how="left"
)
# 定义计数列名列表
count_columns = ["商品详情cnt", "加入购物车cnt", "唤起购买cnt", "首屏总点击cnt", "总点击cnt"]
# 遍历计数列
for col in count_columns:
    # 将空值填充为0并转换为整数类型
    all_data_df[col] = all_data_df[col].fillna(0).astype(int)

# 创建 "用户是否点击" 列
all_data_df["用户是否点击"] = (all_data_df["总点击cnt"] > 0).astype(int)

# 定义费率计算相关列名列表
rate_columns = [
    ("sku_click_rate", "商品详情cnt", "商品查看cnt"), # 商品详情点击率
    ("add_cart_rate", "加入购物车cnt", "商品查看cnt"), # 加入购物车率
    ("popup_click_rate", "唤起购买cnt", "商品查看cnt"), # 唤起购买率
]

# 遍历费率列
for rate_col, num_col, den_col in rate_columns:
    # 计算费率，空值填充0，保留5位小数，转换为浮点数
    all_data_df[rate_col] = (all_data_df[num_col] / all_data_df[den_col]).fillna(0).round(5).astype(float)

all_data_df.head(5)

,uid,ds,variant_list,搜索频次标签,sample_query,商品查看cnt,查看SKU_cnt,查看搜索词cnt,max查看位置,搜索翻页数cnt,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt,用户是否点击,sku_click_rate,add_cart_rate,popup_click_rate
0,,20260305,None,top400,低筋粉,34,19,5,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
1,,20260305,None,top400以外,绿豆馅,8,4,1,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
2,,20260306,None,top400,奶油奶酪,45,22,6,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
3,,20260306,None,top400以外,冷萃酸奶,21,12,3,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
4,,20260307,None,top400,kiri奶油奶酪,15,11,3,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0


In [22]:
from IPython.core.display import HTML
import pandas as pd

css = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.0.0/dist/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
<style type=\"text/css\">
#abTesting table,#abTesting .table {
    color: #333;
    font-family: unset;
    font-size: 12px;
    line-height: 1.5;
    width: 95vw;
    border-collapse:
    collapse; 
    border-spacing: 0;
    font-family: "SF Pro SC", "SF Pro Text", "SF Pro Icons", "PingFang SC", "Helvetica Neue", "Helvetica", "Arial", sans-serif;
}

body{
    padding-left: 1rem;
    padding-top: 1vh;
}

tr{
    border-bottom: 1px solid #C1C3D1;
}

tr:nth-child(even) {
    background-color: #F8F8F8;
}

#abTesting td, #abTesting th {
    /* border: 1px solid transparent; No more visible border */
    height: 30px;
    padding: 0.2rem;
}

#abTesting table tbody td,#abTesting .table tbody td{
    padding: 0.1rem .75rem;
    vertical-align: middle;
}

th {
    background-color: #DFDFDF; /* Darken header a bit */
    font-weight: bolder;
    font-size: larger;
    color: #000;
    text-align: center;
}
</style>
"""


def display_p_value_below_005(row: pd.Series, p_value_col_name: str = "p_value"):
    p_value = row[p_value_col_name]
    color = "black"
    if p_value is not None and p_value <= 0.05:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{p_value}</span>"""


def display_diff_to_v2(row: pd.Series, metric: str = "diff_to_v2%"):
    diff = row[metric]
    color = "green"
    if diff is not None and float(diff) > 0.0:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{diff:.4f} %</span>"""


def dataframe_to_html(df: pd.DataFrame, title: str):
    df_to_display = df.copy()

    df_to_display["p_value"] = df_to_display.apply(display_p_value_below_005, axis=1)
    df_to_display["diff_to_v2%"] = df_to_display.apply(display_diff_to_v2, axis=1)

    html_df = df_to_display.to_html(
        escape=False, index=False, classes="table dataframe"
    )
    html_content = f"""<html><head><meta charset="UTF-8">
    <meta name="title" content="{title}">
    {css}
    </head><body>
    <h2>{title}</h2>
    <h4>当P-value <= 0.05时表示实验结果统计学显著</h4>
    <span>统计学显著时，既可能表示该试验组是好于对照组，也可能是坏于对照组</span>
    <div id="abTesting">{html_df}</div></body></html>"""

    return html_content

In [23]:
import pandas as pd
from scipy.stats import ttest_ind


def calculate_p_values(
    df: pd.DataFrame,
    metric: str = "商品详情cnt",
    control_variant: str = "V2",
) -> pd.DataFrame:
    """
    Calculate p-values for each combination of category1 and page_name.
    Compares metric between control group (V1) and each of V2, V3, V4.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing A/B test data.
    - metric (str): The metric column to be analyzed (default is 'added_quantity').

    Returns:
    - pd.DataFrame: A DataFrame with category1, page_name, variant, p-value, and statistics columns.
    """
    p_values = []

    control = df[df["variant_list"] == control_variant][metric]
    control_avg = control.mean()

    for variant in ["V1", "V2", "V3", "V4"]:
        test_group = df[df["variant_list"] == variant]
        test = test_group[metric]

        # print(
        #     f'variant:{variant}, test_group ds length: {len(test_group["ds"].unique())}'
        # )
        if len(test_group["ds"].unique()) <= 0:
            continue

        # Calculate statistics
        stats = {
            "均值": round(test.mean(), 4),
            "std": round(test.std(), 4),
            f"diff_to_{control_variant}%".lower(): round(
                100.00 * (test.mean() - control_avg) / control_avg, 2
            ),
            "q50": test.quantile(0.5),
            "q75": test.quantile(0.75),
            "q90": test.quantile(0.9),
            "q95": test.quantile(0.95),
            "q97": test.quantile(0.97),
            "q99": test.quantile(0.99),
            "q995": test.quantile(0.995),
            "max": test.max(),
            "日均总数": round(test.sum() / len(test_group["ds"].unique())),
            "日均实验UV": round(
                len(test_group[["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日均转化UV": round(
                len(test_group[test_group[metric] > 0][["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日期范围": f"{test_group['ds'].min()}~{test_group['ds'].max()}".replace(
                "2025", ""
            ),
            "metric": metric,
        }

        # Ensure both groups have enough data for a valid t-test
        if len(control) > 1 and len(test) > 1:
            # Perform independent t-test
            stat, p_val = ttest_ind(control, test, equal_var=False, nan_policy="omit")
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": round(p_val, 4),
                    **stats,
                }
            )
            # print(f"stat:{stat}")
        else:
            # Not enough data for statistical testing
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": None,
                    **stats,
                }
            )

    return pd.DataFrame(p_values)

In [24]:
# Define the desired order for sorting
variant_order = ["V1", "V2", "V3", "V4"]


# Create a custom sort key function
def sort_key(variant):
    # Split variant by commas
    parts = variant.split(",")
    # Determine the order based on the first variant in the list
    if parts[0] in variant_order:
        return variant_order.index(parts[0])
    else:
        return len(variant_order)  # Place all other variants after V1, V2, V3, V4


metrics_list = [
    "sku_click_rate",
    "avg点击位置",
    "唤起购买cnt",
    "总点击cnt",
    "商品详情cnt",
    "加入购物车cnt",
    "商品查看cnt",
]
all_p_values_df = pd.DataFrame()
for metric in metrics_list:
    all_p_values_of_same_metric_df = pd.DataFrame()
    for label, group_df in all_data_df.groupby("搜索频次标签"):
        if label == "其他":
            # print("ignore 其他")
            continue
        p_values_df = calculate_p_values(
            group_df,
            metric=metric,
        )

        p_values_df["搜索频次"] = label

        p_values_df["variant_list"] = p_values_df["variant_list"].apply(
            lambda x: x if x in variant_order else "X_" + x
        )
        p_values_df = p_values_df.sort_values(
            by="variant_list", key=lambda x: x.map(sort_key)
        )
        p_values_df["variant_list"] = p_values_df["variant_list"].str.replace("X_", "")
        all_p_values_of_same_metric_df = pd.concat(
            [all_p_values_of_same_metric_df, p_values_df], ignore_index=True
        )
        all_p_values_df = pd.concat([all_p_values_df, p_values_df], ignore_index=True)

    title = f"搜索AB--{metric}_p-value分布-{p_values_df.iloc[0]['日期范围']}"

    html_content = dataframe_to_html(df=all_p_values_of_same_metric_df, title=title)
    file_path = f"./data/{title}.html"

    # 保存HTML到本地文件：
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"写入HTML成功！{file_path}")


all_p_values_df = all_p_values_df[['metric', '搜索频次', 'variant_list', 'p_value', '均值', 'std', 'diff_to_v2%', 
         'q50', 'q75', 'q90', 'q95', 'q97', 'q99', 'q995', 'max', '日均总数', 
         '日均实验UV', '日均转化UV', '日期范围']]

title_all = f"搜索AB--指标全集_p-value分布-{all_p_values_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_p_values_df, title=title_all)
file_path = f"./data/{title_all}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
# 根据你的截图，完整的列顺序调整为：
file_path = f"./data/{title_all}.csv"
all_p_values_df.to_csv(file_path, index=False)
all_p_values_df

写入HTML成功！./data/搜索AB--sku_click_rate_p-value分布-20260305~20260321.html
写入HTML成功！./data/搜索AB--avg点击位置_p-value分布-20260305~20260321.html
写入HTML成功！./data/搜索AB--唤起购买cnt_p-value分布-20260305~20260321.html
写入HTML成功！./data/搜索AB--总点击cnt_p-value分布-20260305~20260321.html
写入HTML成功！./data/搜索AB--商品详情cnt_p-value分布-20260305~20260321.html
写入HTML成功！./data/搜索AB--加入购物车cnt_p-value分布-20260305~20260321.html
写入HTML成功！./data/搜索AB--商品查看cnt_p-value分布-20260305~20260321.html
写入HTML成功！./data/搜索AB--指标全集_p-value分布-20260305~20260321.html


,metric,搜索频次,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围
0,sku_click_rate,top400,V1,0.5125,0.0542,0.1082,-1.14,0.0,0.06250,0.18182,0.25000,0.33333,0.500,0.615057,1.75,83,1525,613,20260305~20260321
1,sku_click_rate,top400,V2,1.0000,0.0548,0.1082,0.00,0.0,0.06667,0.20000,0.25000,0.33333,0.500,0.600000,2.00,83,1511,618,20260305~20260321
2,sku_click_rate,top400,V3,0.2319,0.0560,0.1136,2.13,0.0,0.06667,0.20000,0.25000,0.33333,0.500,0.666670,3.50,85,1513,607,20260305~20260321
3,sku_click_rate,top400,V4,0.0256,0.0569,0.1098,3.93,0.0,0.07042,0.20000,0.25000,0.33333,0.500,0.613716,1.50,85,1497,612,20260305~20260321
4,sku_click_rate,top400以外,V1,0.5901,0.0552,0.1206,1.55,0.0,0.05882,0.22222,0.26316,0.33333,0.500,0.666670,2.00,36,649,220,20260305~20260321
5,sku_click_rate,top400以外,V2,1.0000,0.0544,0.1114,0.00,0.0,0.06250,0.20000,0.25000,0.33333,0.500,0.646817,1.50,36,655,233,20260305~20260321
6,sku_click_rate,top400以外,V3,0.2954,0.0560,0.1157,2.94,0.0,0.06522,0.20000,0.25000,0.33333,0.500,0.666670,2.00,36,648,229,20260305~20260321
7,sku_click_rate,top400以外,V4,0.7797,0.0540,0.1086,-0.76,0.0,0.06250,0.20000,0.25000,0.33333,0.500,0.500000,1.75,34,637,228,20260305~20260321
8,avg点击位置,top400,V1,0.0506,1.2637,5.8720,-8.36,-0.8,1.00000,5.00000,10.00000,15.00000,27.000,37.552500,164.60,1509,1525,372,20260305~20260321
9,avg点击位置,top400,V2,1.0000,1.3789,5.9933,0.00,-0.8,1.00000,5.30000,10.77500,16.47000,28.000,37.000000,130.70,1639,1511,382,20260305~20260321


In [25]:
# 导入odps_client库中的两个函数：get_odps_sql_result_as_df 用于执行SQL查询并将结果作为DataFrame返回, write_pandas_df_into_odps 用于将pandas DataFrame写入ODPS表
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps

# 定义分区规范字符串，使用当前日期（年-月-日格式）作为分区值
partition_spec = f"pt={datetime.now().strftime('%Y%m%d')}"

# 将DataFrame写入ODPS表
write_pandas_df_into_odps(
    df=all_user_variant_df,  # 要写入的DataFrame，这里是all_user_variant_df，包含了所有用户的变体信息
    table_name="summerfarm_ds.temp_search_ab_all_data_df",  # ODPS表名
    partition_spec=partition_spec,  # 分区规范
    overwrite=True,  # 如果表或分区已存在，是否覆盖
    lifecycle=30,  # 设置表的生命周期为30天
)

# 只分析哪些进入过搜索页面的用户的订单转化结果

# 将开始日期格式化为字符串（年-月-日）
start_date_str = start_date.strftime("%Y-%m-%d")

# 定义SQL查询字符串，用于获取用户订单数据.
# 这段SQL的目的是：从订单表和用户分流表中，根据用户ID和日期进行关联，
# 统计每个用户在不同实验变体下的订单总金额、订单数量和平均订单金额。
order_query = f"""
with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '{start_date_str} 00:00:00'
    AND     m_size = '单店'
),user_variants as (
    select ds as event_date,uid,variant_list
    from summerfarm_ds.temp_search_ab_all_data_df
    where pt=max_pt('summerfarm_ds.temp_search_ab_all_data_df')
)
select a.event_date,a.uid,a.variant_list,sum(b.total_price) as order_gmv,
    count(b.order_no) as order_cnt,round(sum(b.total_price)/count(b.order_no),2) as avg_order_gmv
from user_variants a
left join user_orders b on a.uid=b.m_id and a.event_date=b.order_date
group by a.event_date,a.uid,a.variant_list
"""

# 执行SQL查询并将结果作为DataFrame返回
user_orders_df = get_odps_sql_result_as_df(order_query)
# 显示DataFrame的前两行
user_orders_df.head(2)

2026-03-22 21:16:24 - INFO - DaraFrame字段合集:api,variant_list,experiment_id,ds,search_times,create_time,page_name,type,pt,page_ame,uid,api_list
2026-03-22 21:16:27 - INFO - Tunnel session created: <TableUploadSession id=20260322211627d3d9c20b306bfdb0 project=summerfarm_ds table=temp_search_ab_all_data_df partition_spec=pt=20260322>
2026-03-22 21:16:34 - INFO - 成功写入odps:summerfarm_ds.temp_search_ab_all_data_df, partition_spec:pt=20260322, attemp:0
2026-03-22 21:16:40 - INFO - Tunnel session created: <InstanceDownloadSession id=20260322211640ae19481a2b90d2bf project_name=summerfarm_ds instance_id=20260322131634364gtydm645eq1>
2026-03-22 21:16:41 - INFO - sql:

with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2026-03-05 00:00:00'
    AND     m_size = '

,event_date,uid,variant_list,order_gmv,order_cnt,avg_order_gmv
0,20260305,100522,V3,991.5,2,495.75
1,20260305,100583,V1,None,0,None


In [26]:
user_orders_df["order_gmv"]=user_orders_df["order_gmv"].astype(float)
user_orders_df["avg_order_gmv"]=user_orders_df["avg_order_gmv"].astype(float)
user_orders_df["order_cnt"]=user_orders_df["order_cnt"].astype(int)
user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].describe()

count    61034.000000
mean       591.108093
std        986.215136
min          0.010000
25%        180.000000
50%        330.000000
75%        653.000000
max      48124.800000
Name: order_gmv, dtype: float64

In [27]:
print(
    f"所有订单的分布:\n",
    user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].quantile(
        [0.5, 0.75, 0.95, 0.99, 0.995, 0.996, 0.997, 0.999, 1]
    ),
)


# 这里排除哪些高单价的订单，否则对于数据分析来说不好处理。
user_orders_below_6k_df = user_orders_df[user_orders_df["order_gmv"] <= 6000]
print(
    "排除高单价的订单后的分布:\n",
    user_orders_below_6k_df["order_gmv"].quantile(
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ),
)

所有订单的分布:
 0.500      330.00000
0.750      653.00000
0.950     1750.00000
0.990     3917.67000
0.995     5750.00000
0.996     6119.46672
0.997     7660.00000
0.999    11960.69100
1.000    48124.80000
Name: order_gmv, dtype: float64
排除高单价的订单后的分布:
 0.01      47.2078
0.05      89.0000
0.25     180.0000
0.50     329.0000
0.75     648.0000
0.95    1677.2440
0.99    3265.0000
Name: order_gmv, dtype: float64


In [28]:
user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].astype(
    float
)

user_orders_below_6k_df["avg_order_gmv"].fillna(0.0, inplace=True)
user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df[
    "avg_order_gmv"
].astype(float)

user_orders_below_6k_df["category1"] = "ignore"
user_orders_below_6k_df["page_name"] = "ignore"

user_orders_during_ab_df = user_orders_below_6k_df[
    user_orders_below_6k_df["variant_list"].isin(["V1", "V2", "V3", "V4"])
]
user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)

all_order_pvalue_df = pd.DataFrame()
for metric in ["order_gmv", "avg_order_gmv", "order_cnt"]:
    gmv_df = calculate_p_values(df=user_orders_during_ab_df, metric=metric)
    display(gmv_df)
    all_order_pvalue_df = pd.concat([all_order_pvalue_df, gmv_df], ignore_index=True)

title = f"搜索AB--订单转化p-value分布-{all_order_pvalue_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_order_pvalue_df, title=title)
file_path = f"./data/{title}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
display(all_order_pvalue_df)

/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_8879/858576852.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_8879/858576852.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_orders_below_6k_df["order_g

,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1104,558.0621,653.9045,2.16,328.0,654.0,1198.000,1760.15,2331.4500,3297.677,4069.8600,5987.0,500877,898,898,20260305~20260321,order_gmv
1,V2,1.0000,546.2713,633.4821,0.00,331.0,646.0,1164.455,1650.00,2127.9505,3343.145,4422.5075,5970.0,486374,890,890,20260305~20260321,order_gmv
2,V3,0.3320,539.2694,623.0000,-1.28,326.0,639.0,1148.000,1640.00,2120.0000,3218.560,4238.2800,5997.0,481314,893,893,20260305~20260321,order_gmv
3,V4,0.6283,549.7544,619.7106,0.64,329.0,655.0,1200.000,1675.00,2133.3400,3199.560,3886.7800,6000.0,491933,895,895,20260305~20260321,order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.7827,450.2469,507.1836,0.36,280.00,553.8325,950.0,1320.3,1656.45,2628.590,3150.000,5926.0,404110,898,898,20260305~20260321,avg_order_gmv
1,V2,1.0000,448.6470,503.9453,0.00,282.00,557.5850,950.0,1298.5,1630.00,2520.510,3201.625,5900.0,399454,890,890,20260305~20260321,avg_order_gmv
2,V3,0.2353,441.8601,491.5647,-1.51,279.95,548.5000,936.0,1267.4,1600.84,2445.600,3100.560,5800.0,394373,893,893,20260305~20260321,avg_order_gmv
3,V4,0.6762,451.0235,487.0834,0.53,283.00,568.0000,961.0,1335.0,1630.00,2373.455,3055.670,5800.0,403586,895,895,20260305~20260321,avg_order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.0068,1.2721,0.6779,1.60,1.0,1.0,2.0,2.0,3.0,4.0,5.0,18,1142,898,898,20260305~20260321,order_cnt
1,V2,1.0000,1.2520,0.6101,0.00,1.0,1.0,2.0,2.0,3.0,4.0,4.0,12,1115,890,890,20260305~20260321,order_cnt
2,V3,0.7832,1.2540,0.6271,0.16,1.0,1.0,2.0,2.0,3.0,4.0,4.0,12,1119,893,893,20260305~20260321,order_cnt
3,V4,0.8214,1.2537,0.6497,0.13,1.0,1.0,2.0,2.0,3.0,4.0,4.0,16,1122,895,895,20260305~20260321,order_cnt


写入HTML成功！./data/搜索AB--订单转化p-value分布-20260305~20260321.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1104,558.0621,653.9045,2.16,328.00,654.0000,1198.000,1760.15,2331.4500,3297.677,4069.8600,5987.0,500877,898,898,20260305~20260321,order_gmv
1,V2,1.0000,546.2713,633.4821,0.00,331.00,646.0000,1164.455,1650.00,2127.9505,3343.145,4422.5075,5970.0,486374,890,890,20260305~20260321,order_gmv
2,V3,0.3320,539.2694,623.0000,-1.28,326.00,639.0000,1148.000,1640.00,2120.0000,3218.560,4238.2800,5997.0,481314,893,893,20260305~20260321,order_gmv
3,V4,0.6283,549.7544,619.7106,0.64,329.00,655.0000,1200.000,1675.00,2133.3400,3199.560,3886.7800,6000.0,491933,895,895,20260305~20260321,order_gmv
4,V1,0.7827,450.2469,507.1836,0.36,280.00,553.8325,950.000,1320.30,1656.4500,2628.590,3150.0000,5926.0,404110,898,898,20260305~20260321,avg_order_gmv
5,V2,1.0000,448.6470,503.9453,0.00,282.00,557.5850,950.000,1298.50,1630.0000,2520.510,3201.6250,5900.0,399454,890,890,20260305~20260321,avg_order_gmv
6,V3,0.2353,441.8601,491.5647,-1.51,279.95,548.5000,936.000,1267.40,1600.8400,2445.600,3100.5600,5800.0,394373,893,893,20260305~20260321,avg_order_gmv
7,V4,0.6762,451.0235,487.0834,0.53,283.00,568.0000,961.000,1335.00,1630.0000,2373.455,3055.6700,5800.0,403586,895,895,20260305~20260321,avg_order_gmv
8,V1,0.0068,1.2721,0.6779,1.60,1.00,1.0000,2.000,2.00,3.0000,4.000,5.0000,18.0,1142,898,898,20260305~20260321,order_cnt
9,V2,1.0000,1.2520,0.6101,0.00,1.00,1.0000,2.000,2.00,3.0000,4.000,4.0000,12.0,1115,890,890,20260305~20260321,order_cnt


In [29]:
all_p_values_df.to_csv(
    f"./data/搜索AB--所有指标p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)
all_order_pvalue_df.to_csv(
    f"./data/搜索AB--订单转化p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)